# 10초 AF/비AF 실행 확인 — MIMIC 비교 기준
**세 자료 통합 학습이 아닙니다.** 현재 실행 대상은 MIMIC 35명뿐입니다.
6종의 사용 범위, 손목 v3의 정답 코드·시간 연결이 미확정입니다.
10초 × 50Hz = 500 샘플로 재학습하고 참가자별 시험을 수행합니다.
기존 시험 참가자를 다시 평가하므로 새로운 외부 최종 검증이 아닙니다.
로컬 CPU에서 실행한 결과는 저장소 `models/af_mimic_10s_v1/`에 있으며 이 노트북 자체의 Colab 실행 완료를 뜻하지 않습니다.
GPU 선택 후 아래 셀을 순서대로 실행하세요. 결과는 임시 저장소이므로 마지막 셀에서 내려받으세요.


In [ ]:
from pathlib import Path
import subprocess, sys, json
from datetime import datetime, timezone
REPO=Path('/content/capdyssey_10s')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-train.txt')],check=True)
print(json.loads((REPO/'data/af_integration_status.json').read_text()))


In [ ]:
subprocess.run([sys.executable,str(REPO/'scripts/download_data.py'),'mimic_af'],check=True)
RUN=REPO/'runs'/('mimic10_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
subprocess.run([sys.executable,'-u',str(REPO/'scripts/train_af.py'),'--seconds','10','--epochs','20','--seed','42','--output',str(RUN)],check=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
report=json.loads((RUN/'metrics.json').read_text())
print(json.dumps(report,indent=2))
m=report['test_window_metrics']
matrix=np.array([[m['tn'],m['fp']],[m['fn'],m['tp']]])
fig,ax=plt.subplots()
ax.imshow(matrix,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(matrix[i,j]),ha='center',va='center')
ax.set_xticks([0,1],['non-AF','AF']); ax.set_yticks([0,1],['non-AF','AF'])
ax.set_xlabel('Prediction'); ax.set_ylabel('Reference')
ax.set_title('MIMIC only / 10 seconds / not three-dataset integrated')
plt.show()


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/'+RUN.name,'zip',RUN)
files.download(archive)
